<a href="https://colab.research.google.com/github/PabloVM/JLM-jev_like_model/blob/main/train/jlm-train-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import os
REPO_URL = "https://github.com/PabloVM/JLM-jev_like_model"
REPO_NAME = "JLM-jev_like_model"
!git clone {REPO_URL}





sys.path.append(f"{REPO_NAME}")


Cloning into 'JLM-jev_like_model'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 79 (delta 31), reused 62 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 265.17 KiB | 1.41 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [ ]:
%cd /kaggle/working/JLM-jev_like_model
!git pull

In [2]:
from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token

In [ ]:
import torch
import gc
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from pathlib import Path

from src.data import JLMTokenizer, JLMDataset
from src.model import JLM
from src.cache import build_embedding_cache, iterate_cached_batches, cache_to

In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"
CACHE_BATCH_SIZE = 32   # batch for the single backbone pass (lower it on OOM)
BATCH_SIZE = 64         # batch for training the head on cached embeddings
LEARNING_RATE = 1e-4
NUM_EPOCHS = 10         # epochs are now cheap: only the head runs
NUM_WORKERS = 2
SAVE_DIR = Path("./checkpoints")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path("/kaggle/working/cache")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [5]:
print("Loading Dataset : 'tasksource/tasksource-jev-typed-decisions'...")
train_hf = load_dataset("tasksource/tasksource-jev-typed-decisions", split="train")
test_hf = load_dataset("tasksource/tasksource-jev-typed-decisions", split="test")

def filter_long_examples(example):
    options = example.get("options") or []
    return len(options) <= 6 and len(example.get("question", "")) < 1000

train_hf = train_hf.filter(filter_long_examples)
test_hf = test_hf.filter(filter_long_examples)

train_hf = train_hf.shuffle(seed=42).select(range(50000))
val_hf = test_hf.shuffle(seed=42).select(range(1000))

print(f"Train entries : {len(train_hf)} | Validation entries : {len(val_hf)}")

Loading Dataset : 'tasksource/tasksource-jev-typed-decisions'...


README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

data/train-00000-of-00006.parquet: reconstructing file:   0%|          |  0.00B /  112MB            

data/train-00000-of-00006.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00006.parquet: reconstructing file:   0%|          |  0.00B /  112MB            

...

Generating validation split:   0%|          | 0/15000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/15000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/999832 [00:00<?, ? examples/s]

Filter:   0%|          | 0/15000 [00:00<?, ? examples/s]

Train entries : 50000 | Validation entries : 1000


In [ ]:
tokenizer = JLMTokenizer(MODEL_NAME)

train_dataset = JLMDataset(train_hf, tokenizer)
val_dataset = JLMDataset(val_hf, tokenizer)

In [ ]:
model = JLM(MODEL_NAME)
model.to(device)

print(f"GPU : {torch.cuda.get_device_name(device)} {torch.cuda.get_device_capability(device)}")
print(f"Backbone dtype : {model.backbone_dtype}")

## Embedding cache
The backbone is frozen, so its embeddings never change. We run Qwen **once** over train and validation,
store the answer/choice embeddings and train the head on them. The cache is saved to `CACHE_DIR`
and reloaded if it already exists (delete it after changing the tokenizer or the data filter).

In [ ]:
train_cache = build_embedding_cache(
    model, train_dataset, batch_size=CACHE_BATCH_SIZE, num_workers=NUM_WORKERS,
    device=device, path=CACHE_DIR / "train_50k.pt",
)
val_cache = build_embedding_cache(
    model, val_dataset, batch_size=CACHE_BATCH_SIZE, num_workers=NUM_WORKERS,
    device=device, path=CACHE_DIR / "val_1k.pt",
)

# ~0.7 GB in fp16: fits on the GPU, so batches are sliced without CPU->GPU copies
train_cache = cache_to(train_cache, device)
val_cache = cache_to(val_cache, device)
gc.collect()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.head.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

In [ ]:
def train_epoch(model, cache, optimizer, criterion, batch_size):
    model.head.train()
    total_loss, correct, total, num_batches = 0.0, 0, 0, 0

    for answer_embed, choice_embeds, choice_mask, labels in iterate_cached_batches(
        cache, batch_size, shuffle=True
    ):
        optimizer.zero_grad()

        # logits (batch_size, num_choices)
        logits = model.score(answer_embed, choice_embeds, choice_mask)

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        pred = torch.argmax(logits, dim=-1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    return total_loss / num_batches, (correct / total) * 100

@torch.no_grad()
def evaluate(model, cache, criterion, batch_size):
    model.head.eval()
    total_loss, correct, total, num_batches = 0.0, 0, 0, 0

    for answer_embed, choice_embeds, choice_mask, labels in iterate_cached_batches(
        cache, batch_size, shuffle=False
    ):
        logits = model.score(answer_embed, choice_embeds, choice_mask)
        loss = criterion(logits, labels)
        total_loss += loss.item()
        num_batches += 1

        pred = torch.argmax(logits, dim=-1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    return total_loss / num_batches, (correct / total) * 100

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
HF_REPO_ID = "PabloVM/JLM"

best_val_acc = 0.0

print("\n=================== START TRAINING ===================")
for epoch in range(0, NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_cache, optimizer, criterion, BATCH_SIZE)
    val_loss, val_acc = evaluate(model, val_cache, criterion, BATCH_SIZE)

    print(f"Epoch {epoch}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.2f}%")


    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_path = SAVE_DIR / "best_head.pt"
        torch.save(model.head.state_dict(), save_path)
        print(f"New best validation accuracy. Weights saved: {save_path}")

        try:
          print("Uploading file to Hugging Face Hub...")
          api.upload_file(
              path_or_fileobj=save_path,
              path_in_repo="best_head.pt",
              repo_id=HF_REPO_ID,
              repo_type="model",
          )
          print("File uploaded to Hugging Face Hub")
        except Exception as e:
          print(f"Error uploading file to Hugging Face Hub: {e}")

print("=================== END TRAINING ===================")
print(f"Best validation accuracy: {best_val_acc:.2f}%")